# Forge: a short campaign

The same window and covariance as `block.ipynb`, a `Wheel`, the block
registered on its grid, and a chain of its parameters collected through
`on_cycle_complete`. A second block goes where the comment says.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import numpy as np

from block import ForgeBlock
from data import analytic_covariance, load_window, white_psd
from enchilada import Wheel, transform

MOJITO_PATH = os.environ.get("ENCHILADA_MOJITO_L1")
if not MOJITO_PATH:
    raise SystemExit("set ENCHILADA_MOJITO_L1 to a Mojito-layout L1 file")
DURATION_S = float(os.environ.get("FORGE_DURATION_S", 86400.0))
FACTOR = int(os.environ.get("FORGE_FACTOR", 10))
DOMAIN = "frequency"  # the grid ForgeBlock renders on; see _check_residual
CHANNELS = ("A", "E")
TRUE_AMPLITUDE = None  # the injected amplitude, when you know it

window = load_window(
    MOJITO_PATH, factor=FACTOR, duration_s=DURATION_S, channels=CHANNELS
)
# A white stand-in for the simulation's noise model, at the window's own level
# (about 12 % below the in-band PSD: decimation rolls off the top fifth of the band).
# For the LISA PSD: `from gb_model import FixedLISANoise` (examples/, needs
# lisatools) and `PSD = FixedLISANoise().psd`.
LEVEL = 2 * np.mean([window.channel_data[name].var() for name in CHANNELS]) / window.fs
PSD = white_psd(LEVEL)
residual = transform(window, DOMAIN)
covariance = analytic_covariance(residual, PSD)

In [ ]:
def rms(data):
    return float(
        np.sqrt(np.mean([np.abs(data.channel_data[name]) ** 2 for name in CHANNELS]))
    )


wheel = Wheel(residual, initial_covariance=covariance, random_seed=0)
block = ForgeBlock(name="forge", frequency_hz=1e-3)
wheel.add(block, data_domain=DOMAIN)
# A second block goes here, registered on its own grid, for example:
# wheel.add(OtherBlock(name="other"), data_domain="wdm", num_frequency_divisions=512)
print(f"residual rms after the starts {rms(wheel.residual()):.3e}")

In [ ]:
NUM_CYCLES = 200
WARM_UP = 0  # the placeholder draws the posterior exactly; an MCMC sampler needs some
chain = []


def collect(cycle_index, campaign):
    chain.append(campaign.ledger["forge"].parameters["amplitude"])


wheel.run(NUM_CYCLES, on_cycle_complete=collect)
samples = np.asarray(chain[WARM_UP:])
print(
    f"posterior amplitude: {samples.mean():+.3e} +/- {samples.std():.3e} "
    f"over {samples.size} cycles"
)
if TRUE_AMPLITUDE is not None:
    offset = (samples.mean() - TRUE_AMPLITUDE) / samples.std()
    print(
        f"truth {TRUE_AMPLITUDE:+.3e}: the posterior mean is {offset:+.2f} sigma away"
    )
print(f"residual rms after the run    {rms(wheel.residual()):.3e}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None:
    figure, axes = plt.subplots(figsize=(8, 3))
    axes.plot(samples)
    if TRUE_AMPLITUDE is not None:
        axes.axhline(TRUE_AMPLITUDE, color="k", linestyle="--")
    axes.set_xlabel("cycle")
    axes.set_ylabel("amplitude")
    plt.show()

## Next

- Replace the placeholder model in `block.py` and rerun both notebooks.
- Register a noise block before the signal blocks whose starts need its
  covariance; blocks start in registration order.
- Run independent campaigns with fresh block instances and different seeds.
- Checkpointing and restarts are outside the forge.